# C4 — Tester ses fonctions

**Objectif** : écrire des tests unitaires pour `valider_mouvement()` et `creer_drone()`
sans avoir besoin de lancer le jeu.

Ce notebook est **100 % autonome** — toutes les données nécessaires sont définies
directement dans les cellules (compatible Google Colab, sans import de fichiers locaux).

## Rappels

| Notion | Résumé |
|---|---|
| `assert condition` | Lève `AssertionError` si `condition` est False |
| `assert condition, message` | Idem + affiche `message` à l'échec |
| Patron AAA | **Arrange** → **Act** → **Assert** |
| Test cas valide | La fonction doit accepter → vérifier `ok == True` |
| Test cas invalide | La fonction doit refuser → vérifier `ok == False` |

**Rappel module précédent (C3)** : un commit = un état stable.
Après avoir écrit tes tests, fais un commit : `git commit -m "C4 : tests valider_mouvement"`.

## Mise en place — Fonctions simulées (pas d'import de fichiers locaux)

In [ ]:
# --- Constantes (reproduites depuis config.json) ---
GRILLE_TAILLE = 12
BATTERIE_INIT = 15
COUT_ZONE_X = 2
COUT_TRANSPORT = 2

# --- Factory drone (version minimale de jeu/logique.py) ---
def creer_drone(drone_id, col, lig, batterie_max=BATTERIE_INIT):
    return {
        'id': drone_id,
        'col': col, 'lig': lig,
        'batterie': batterie_max,
        'batterie_max': batterie_max,
        'survivant': None,
        'bloque': 0,
        'hors_service': False,
    }

# --- Validation de mouvement (version simplifiée pour les exercices) ---
def valider_mouvement(etat, drone, cible):
    """Retourne (bool, str). Version pédagogique."""
    col_c, lig_c = cible
    # Hors service ?
    if drone['hors_service']:
        return False, 'drone hors service'
    # Bloqué ?
    if drone['bloque'] > 0:
        return False, 'drone bloqué'
    # Hors grille ?
    if not (0 <= col_c < GRILLE_TAILLE and 0 <= lig_c < GRILLE_TAILLE):
        return False, 'cible hors grille'
    # Distance Chebyshev > 1 ?
    if max(abs(col_c - drone['col']), abs(lig_c - drone['lig'])) > 1:
        return False, 'déplacement trop grand'
    # Bâtiment ?
    if cible in etat.get('batiments', []):
        return False, 'case bâtiment'
    # Batterie suffisante ?
    cout = 1
    if drone.get('survivant'):
        cout = COUT_TRANSPORT
    if cible in etat.get('zones_x', set()):
        cout += COUT_ZONE_X
    if drone['batterie'] < cout:
        return False, f'batterie insuffisante (coût={cout}, batterie={drone["batterie"]}'
    return True, 'ok'

# --- État minimal reproductible ---
def etat_minimal(batiments=None, zones_x=None):
    return {
        'batiments': batiments or [],
        'zones_x': zones_x or set(),
        'hopital': (0, 0),
    }

print('Fonctions chargées.')

## Étape 1 — Tester `creer_drone()` : vérifier la structure
**Objectif** : s'assurer que `creer_drone()` retourne bien un dictionnaire
avec toutes les clés attendues et les bonnes valeurs initiales.

In [ ]:
# ARRANGE
drone = creer_drone('D1', col=5, lig=3)

# ACT + ASSERT — vérifier toutes les clés
assert drone['id'] == 'D1', f"id attendu 'D1', got {drone['id']}"
assert drone['col'] == 5
assert drone['lig'] == 3
assert drone['batterie'] == BATTERIE_INIT
assert drone['survivant'] is None
assert drone['bloque'] == 0
assert drone['hors_service'] is False

# TODO : ajoute un assert pour vérifier que 'batterie_max' == BATTERIE_INIT
# assert ...

print('Test creer_drone : OK')

## Étape 2 — Cas valide : déplacement normal
**Objectif** : vérifier qu'un déplacement d'une case est accepté.

In [ ]:
# ARRANGE
etat = etat_minimal()
drone = creer_drone('D1', col=3, lig=3)

# ACT
ok, raison = valider_mouvement(etat, drone, cible=(4, 3))

# ASSERT
assert ok, f'Attendu True, got False : {raison}'

print(f'Cas valide (déplacement normal) : {ok} — {raison}')

## Étape 3 — Cas invalide : drone hors service
**Objectif** : vérifier qu'un drone `hors_service=True` ne peut pas se déplacer.

In [ ]:
# ARRANGE
etat = etat_minimal()
drone = creer_drone('D1', col=3, lig=3)
drone['hors_service'] = True  # simulation du cas limite

# ACT
ok, raison = valider_mouvement(etat, drone, cible=(4, 3))

# ASSERT
assert not ok, 'Un drone hors service ne devrait pas pouvoir se déplacer'
assert 'hors service' in raison, f'Message inattendu : {raison}'

print(f'Cas invalide (hors service) : ok={ok} — {raison}')

## Étape 4 — Cas invalide : batterie insuffisante en zone X
**Objectif** : vérifier qu'un drone avec 1 batterie est refusé
si la cible est une zone X (coût = 1 + 2 = 3).

In [ ]:
# ARRANGE
cible = (4, 3)
etat = etat_minimal(zones_x={cible})   # la case cible est une zone X
drone = creer_drone('D1', col=3, lig=3)
drone['batterie'] = 1                  # insuffisant pour coût 3

# ACT
ok, raison = valider_mouvement(etat, drone, cible)

# ASSERT
assert not ok, 'Batterie 1 insuffisante pour zone X (coût=3)'
assert 'insuffisant' in raison.lower(), f'Message inattendu : {raison}'

print(f'Cas invalide (batterie zone X) : ok={ok} — {raison}')

## Étape 5 — Écrire tes propres tests : 2 cas à compléter
**TODO** : complete les deux tests ci-dessous.
- **Test A** : déplacement de 2 cases → doit être refusé
- **Test B** : drone bloqué (`bloque=1`) → doit être refusé

In [ ]:
# ===== TEST A : déplacement de 2 cases =====
etat = etat_minimal()
drone = creer_drone('D1', col=0, lig=0)

# TODO : appelle valider_mouvement avec une cible à 2 cases de distance
# ok, raison = valider_mouvement(etat, drone, cible=(...))
# assert not ok

# ===== TEST B : drone bloqué =====
etat = etat_minimal()
drone = creer_drone('D1', col=3, lig=3)

# TODO : mets drone['bloque'] à 1 et vérifie que le mouvement est refusé
# drone['bloque'] = ...
# ok, raison = valider_mouvement(etat, drone, cible=(4, 3))
# assert ...

print('Tests étape 5 : à compléter')

## Récapitulatif

Tu as écrit **5 tests** couvrant :
- ✅ structure de `creer_drone()`
- ✅ 1 cas valide (déplacement normal)
- ✅ 3 cas invalides (hors service, batterie insuffisante, à compléter)

**Prochaine étape** : ouvre `tests/test_logique.py` dans le repo pour voir
25 tests réels couvrant aussi les coûts, les collisions tempête et la fin de partie.

Lance-les avec : `pytest tests/test_logique.py -v`